# Figure 2 — Differentiation trajectory (pseudotime)

Diffusion pseudotime along the NEPC-N → NEPC-A axis (root = least-differentiated
cell by Doxo1 signature, set in `00_Prepare_data.ipynb`), the Doxo1 and CDKN1A
differentiation readouts, and which perturbations push cells toward
differentiation. Perturbation labels are collapsed so 'ASCL1+NTC' counts as the
single KO 'ASCL1' (`perturbation_clean`). Reads `Data/PaperFigures_combo.h5ad`.

In [ ]:
import _figutils as fu
import importlib; importlib.reload(fu)  # pick up edits to _figutils without kernel restart
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from scipy.stats import ks_2samp, gaussian_kde, fisher_exact
from statsmodels.stats.multitest import multipletests

fu.set_theme()
adata = fu.load("processed")
fu.set_state_categories(adata)
assert "dpt_pseudotime" in adata.obs, "Run 00_Prepare_data.ipynb first."

FDR_THRESHOLD = 0.01
MIN_CELLS = 30
DIFF_STATES = fu.DIFFERENTIATED_STATES   # ["NEPC-A-1","NEPC-A-2"] (Fisher enrichment target)
TOP_N = 8                  # cap the density plots to the N most significant perturbations
DOXO1_FDR = 0.3            # FDR for "increased Doxo1 in Differentiated-1/-2" (MWU, as in 01_Data_outlook)
ENRICH_FDR = 0.01           # FDR for "enriched in the differentiated state" (Fisher)
DIFF_SUBSTATES = ["Differentiated_1", "Differentiated-2"]  # sub-states for the Doxo1 MWU test
print(adata.shape, "| root cell:", adata.uns.get("iroot"))

### a — UMAP coloured by diffusion pseudotime

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color="dpt_pseudotime", color_map="viridis", size=3,
           ax=ax, show=False, frameon=False, title="Diffusion pseudotime")
fu.savefig("Fig2a_umap_pseudotime", fig)

### a1 — Pseudotime backbone: the main path through the UMAP

The UMAP coloured by diffusion pseudotime, with the single trajectory path drawn
on top. Cells are sliced into equal-size pseudotime windows; the median UMAP
position of each ordered slice is joined into one curve from the **NEPC-N root (●)**
to the **NEPC-A end (★)**, with arrowheads pointing root → end. This makes the
pseudotime axis used by every other panel *visible* as a route across the UMAP.

In [ ]:
# Pseudotime backbone: bin cells by pseudotime, take each ordered slice's median
# UMAP position, and join them into the main root->end path. The UMAP is coloured
# by pseudotime; the backbone is itself coloured by pseudotime (white halo + black
# arrowheads keep it readable), showing the direction ● root -> ★ end.
fig, ax = plt.subplots(figsize=(5.5, 5))
_, info = fu.plot_pseudotime_backbone(
    adata, ax=ax, color_by="pseudotime", line_color=None, arrow_color="black",
    n_bins=30, smooth=1, n_arrows=6)
fig.colorbar(info["cbar"], ax=ax, shrink=0.7, pad=0.02, label="Diffusion pseudotime")
ax.set_title("Pseudotime backbone (● root → ★ end)")
fu.savefig("Fig2a1_pseudotime_backbone", fig)

### a2 — Doxo1 differentiation score (defines the trajectory root)

UMAP + trend along pseudotime (day04 vs day10). Should rise NEPC-N → NEPC-A,
validating the Doxo1-rooted pseudotime. Per-cell score exported to `figures/`.

In [ ]:
# UMAP coloured by the Doxo1 differentiation score.
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color="Doxo1program_score", color_map="magma", size=3,
           ax=ax, show=False, frameon=False, vmin="p1", vmax="p99",
           title="Doxo1 differentiation score")
fu.savefig("Fig2a2_umap_doxo1", fig)

# Save the score (with pseudotime / state / perturbation) for downstream use.
adata.obs[["Doxo1program_score", "dpt_pseudotime", fu.CELL_STATE_COL,
           "perturbation_clean", "time_point"]].to_csv(
    fu.FIG_DIR / "Fig2_doxo1_pseudotime_percell.csv")

# Doxo1 score vs pseudotime: rolling-window local mean +/- SD, one line per time point.
pt_all = adata.obs["dpt_pseudotime"].values
score_all = adata.obs["Doxo1program_score"].values
tp_all = adata.obs["time_point"].astype(str).values

fig, ax = plt.subplots(figsize=(7, 4))
fu.shade_state_regions(ax, adata)
for tp in fu.TIMEPOINT_ORDER:
    m = tp_all == tp
    gx, gm, gs = fu.rolling_trend(pt_all[m], score_all[m], frac=0.10)
    col = fu.TIME_PALETTE[tp]
    ax.plot(gx, gm, color=col, lw=2, label=tp)
    ax.fill_between(gx, gm - gs, gm + gs, color=col, alpha=0.15, lw=0)
ax.set_xlabel("Pseudotime"); ax.set_ylabel("Doxo1 differentiation score")
ax.set_title("Doxo1 score along pseudotime"); ax.set_xlim(pt_all.min(), pt_all.max())
ax.legend(frameon=False, fontsize=8, title="Time point", loc="upper left")
fu.savefig("Fig2a3_doxo1_vs_pseudotime", fig)

### a4 — CDKN1A (p21) expression

Log-normalised CDKN1A (from `.raw`) on the UMAP and along pseudotime (day04 vs
day10). p21 marks cell-cycle arrest / differentiation, expected to rise toward NEPC-A.

In [ ]:
gene = "CDKN1A"
assert gene in adata.raw.var_names, f"{gene} not found in adata.raw.var_names"

fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color=gene, use_raw=True, color_map="magma", size=3,
           ax=ax, show=False, frameon=False, vmin="p1", vmax="p99",
           title="CDKN1A (p21) expression")
fu.savefig("Fig2a4_umap_cdkn1a", fig)

xcd = adata.raw[:, gene].X
cdkn1a = xcd.toarray().ravel() if sp.issparse(xcd) else np.asarray(xcd).ravel()

pt_all = adata.obs["dpt_pseudotime"].values
tp_all = adata.obs["time_point"].astype(str).values

fig, ax = plt.subplots(figsize=(7, 4))
fu.shade_state_regions(ax, adata)
for tp in fu.TIMEPOINT_ORDER:
    m = tp_all == tp
    gx, gm, gs = fu.rolling_trend(pt_all[m], cdkn1a[m], frac=0.10)
    col = fu.TIME_PALETTE[tp]
    ax.plot(gx, gm, color=col, lw=2, label=tp)
    ax.fill_between(gx, gm - gs, gm + gs, color=col, alpha=0.15, lw=0)
ax.set_xlabel("Pseudotime"); ax.set_ylabel("CDKN1A expression (log-norm)")
ax.set_title("CDKN1A along pseudotime"); ax.set_xlim(pt_all.min(), pt_all.max())
ax.legend(frameon=False, fontsize=8, title="Time point", loc="upper left")
fu.savefig("Fig2a5_cdkn1a_vs_pseudotime", fig)

### b — Pseudotime distribution per cell state (validates ordering)

In [ ]:
order = fu.ordered_states(adata)
fig, ax = plt.subplots(figsize=(5, 4))
sns.violinplot(data=adata.obs, x=fu.CELL_STATE_COL, y="dpt_pseudotime",
               order=order, palette=[fu.STATE_PALETTE[s] for s in order],
               cut=0, inner="box", ax=ax)
ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_xlabel(""); ax.set_ylabel("Pseudotime"); ax.set_title("Pseudotime by cell state")
fu.savefig("Fig2b_pseudotime_by_state", fig)

### Selection — two criteria, computed separately for day04 and day10

**(1) Perturbation shifts** — enrichment of each perturbation in each of the 6 cell states vs NTC (one-sided Fisher, OR>1, FDR<0.1); shown in the heatmap (Fig2c) and as pseudotime density of the differentiated-enriched perturbations (Fig2d).

**(2) Doxo1 increase vs NTC** — single-cell one-sided **Wilcoxon** on the Doxo1 score, tested separately per timepoint x cell state; perturbations significant in ≥1 test are shown along pseudotime (Fig2e).

In [ ]:
from scipy.stats import fisher_exact

# (1) PERTURBATION SHIFTS: enrichment of each perturbation in EACH of the 6 cell
# states vs NTC (one-sided Fisher, OR>1), per day, BH FDR<0.01. Replaces the KS test.
ENR_FDR = 0.1
LOG2OR_CAP = 6.0

def state_enrichment(obs, states=None, pert_col="perturbation_clean",
                     state_col=fu.CELL_STATE_COL, ref=fu.NTC_LABEL, min_cells=MIN_CELLS):
    states = states or fu.STATE_ORDER
    st = obs[state_col].astype(str).values
    grp = obs[pert_col].astype(str).values
    ntc = grp == ref
    rows = []
    for s in states:
        c = int(((st == s) & ntc).sum()); d = int(ntc.sum() - c)
        for p in sorted(set(grp) - {ref}):
            mp = grp == p
            if mp.sum() < min_cells:
                continue
            a = int(((st == s) & mp).sum()); b = int(mp.sum() - a)
            orr, pv = fisher_exact([[a, b], [c, d]], alternative="greater")
            rows.append({"perturbation": p, "state": s, "n_in_state": a,
                         "n_pert": int(mp.sum()), "odds_ratio": orr, "pval": pv})
    r = pd.DataFrame(rows)
    r["fdr"] = multipletests(r["pval"], method="fdr_bh")[1] if len(r) else []
    with np.errstate(divide="ignore"):
        r["log2_or"] = np.clip(np.log2(r["odds_ratio"].replace(0, np.nan)),
                               -LOG2OR_CAP, LOG2OR_CAP)
    return r

enr_tables, enrich_by_day = {}, {}
for day in fu.TIMEPOINT_ORDER:
    sub = adata.obs[adata.obs["time_point"].astype(str) == day]
    r = state_enrichment(sub)
    r.to_csv(fu.FIG_DIR / f"Fig2_state_enrichment_{day}.csv", index=False)
    enr_tables[day] = r
    # density plot uses perturbations enriched in the DIFFERENTIATED states.
    diff = r[(r.state.isin(fu.DIFFERENTIATED_STATES)) & (r.odds_ratio > 1) & (r.fdr < ENR_FDR)]
    enrich_by_day[day] = diff.sort_values("fdr")["perturbation"].drop_duplicates().tolist()
    print(f"{day}: {(r.fdr < ENR_FDR).sum()} significant pert x state enrichments (FDR<{ENR_FDR}); "
          f"{len(enrich_by_day[day])} perturbations enriched in differentiated states")

# (2) Perturbations INCREASING the Doxo1 score vs NTC: single-cell one-sided
# Wilcoxon, run SEPARATELY per timepoint x cell state (BH within each test);
# a perturbation is kept if significant in >=1 test. min_cells=3 so the small
# differentiated sub-states (where the signal lives) are not excluded.
WILCOX_FDR = 0.1
doxo_sig, doxo_tests = fu.wilcoxon_doxo1_state_time(
    adata, score="Doxo1program_score", fdr=WILCOX_FDR, min_cells=3, require_increase=True)
doxo_tests.to_csv(fu.FIG_DIR / "Fig2_doxo1_wilcoxon_state_time.csv", index=False)
print(f"{len(doxo_sig)} perturbations with Doxo1 > NTC (Wilcoxon FDR<{WILCOX_FDR}) "
      f"in >=1 timepoint x state: {doxo_sig}")

### c — Perturbation enrichment across the 6 cell states (Fisher, by day)

log2 odds ratio of each enriched perturbation in each state vs NTC; `*` marks FDR<0.01. Perturbations ordered by their differentiated-state enrichment.

In [ ]:
import seaborn as sns

def enrichment_heatmap(r, day):
    perts = r.loc[r.fdr < ENR_FDR, "perturbation"].unique()
    if len(perts) == 0:
        print(f"{day}: no significant enrichments"); return
    sub = r[r.perturbation.isin(perts)]
    piv = sub.pivot_table(index="perturbation", columns="state", values="log2_or").reindex(columns=fu.STATE_ORDER)
    fdr = sub.pivot_table(index="perturbation", columns="state", values="fdr").reindex(columns=fu.STATE_ORDER)
    order = piv[fu.DIFFERENTIATED_STATES].max(axis=1).sort_values(ascending=False).index
    piv, fdr = piv.reindex(order), fdr.reindex(order)
    annot = np.where(fdr.values < ENR_FDR, "*", "")
    vmax = np.nanmax(np.abs(piv.values)) or 1.0
    fig, ax = plt.subplots(figsize=(3.6, max(2.5, len(piv) * 0.22 + 1.0)))
    sns.heatmap(piv, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, annot=annot, fmt="",
                linewidths=0.4, linecolor="white", cbar_kws={"label": "log2 odds ratio"}, ax=ax)
    ax.set_title(f"Perturbation enrichment across states — {day}  (* FDR<{ENR_FDR})", fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel("Perturbation")
    plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(rotation=0, fontsize=8)
    fu.savefig(f"Fig2c_state_enrichment_heatmap_{day}", fig)

for day in fu.TIMEPOINT_ORDER:
    enrichment_heatmap(enr_tables[day], day)

### c2 — Timepoint composition across cell states

Left: for each timepoint, the percentage of its cells in each of the 6 states (bars within a timepoint sum to 100%). Right: for each state, the percentage of its cells coming from day04 vs day10 (each bar sums to 100%).

In [ ]:
order = fu.ordered_states(adata)
ct = (pd.crosstab(adata.obs["time_point"], adata.obs[fu.CELL_STATE_COL])
      .reindex(index=fu.TIMEPOINT_ORDER, columns=order))
within_day = ct.div(ct.sum(axis=1), axis=0) * 100      # rows sum to 100 (composition per timepoint)
within_state = ct.div(ct.sum(axis=0), axis=1) * 100    # columns sum to 100 (day share per state)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (1) x = timepoint, stacked & coloured by cell state (each bar = 100%).
ax = axes[0]
xd = np.arange(len(fu.TIMEPOINT_ORDER))
bottom = np.zeros(len(fu.TIMEPOINT_ORDER))
for s in order:
    vals = within_day.loc[fu.TIMEPOINT_ORDER, s].values
    ax.bar(xd, vals, width=0.6, bottom=bottom, color=fu.STATE_PALETTE[s],
           label=s, edgecolor="white", lw=0.4)
    bottom += vals
ax.set_xticks(xd); ax.set_xticklabels(fu.TIMEPOINT_ORDER)
ax.set_ylim(0, 100); ax.set_ylabel("% of the timepoint's cells")
ax.set_title("Cell-state composition per timepoint")
ax.legend(title="Cell state", frameon=False, fontsize=7, ncol=3,
          loc="upper center", bbox_to_anchor=(0.5, -0.12))

# (2) x = state, stacked & coloured by timepoint (each bar = 100%).
ax = axes[1]
xs = np.arange(len(order))
bottom = np.zeros(len(order))
for day in fu.TIMEPOINT_ORDER:
    vals = within_state.loc[day, order].values
    ax.bar(xs, vals, width=0.7, bottom=bottom, color=fu.TIME_PALETTE[day],
           label=day, edgecolor="white", lw=0.4)
    bottom += vals
ax.set_xticks(xs); ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_ylim(0, 100); ax.set_ylabel("% of the state's cells")
ax.set_title("Timepoint composition within each cell state")
ax.legend(title="Time point", frameon=False, fontsize=8)

fu.savefig("Fig2c_timepoint_state_barplots", fig)

### d — Pseudotime density of differentiated-enriched perturbations (day04 | day10)

Perturbations enriched in the differentiated states (NEPC-A-1/-2) vs NTC by the per-state Fisher test above, shown as pseudotime density overlaid on NTC, one plot per day.

In [ ]:
pt = adata.obs["dpt_pseudotime"].values
grp = adata.obs["perturbation_clean"].astype(str).values
tp = adata.obs["time_point"].astype(str).values
xg = np.linspace(pt.min(), pt.max(), 400)
kde = lambda v: gaussian_kde(v, bw_method="scott")(xg) if len(v) >= 10 else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True, sharey=True)
for ci, day in enumerate(fu.TIMEPOINT_ORDER):
    ax = axes[ci]; fu.shade_state_regions(ax, adata, legend=(ci == 1))
    dmask = tp == day
    dn = kde(pt[(grp == fu.NTC_LABEL) & dmask])
    if dn is not None:
        ax.fill_between(xg, dn, color="#bdbdbd", alpha=0.4, label="NTC")
        ax.plot(xg, dn, color="#757575", lw=2)
    perts = enrich_by_day[day]
    cmap = plt.cm.get_cmap("tab20", max(len(perts), 1))
    for i, p in enumerate(perts):
        dp = kde(pt[(grp == p) & dmask])
        if dp is not None:
            ax.plot(xg, dp, color=cmap(i), lw=1.6, label=p)
    ax.set_title(day, fontsize=11); ax.set_xlabel("Pseudotime"); ax.set_xlim(xg[0], xg[-1])
    ax.legend(fontsize=7, frameon=False, loc="upper right")
axes[0].set_ylabel("Density")
fig.suptitle("Pseudotime density vs NTC — perturbations enriched in differentiated states, by day", fontsize=11)
fu.savefig("Fig2d_pseudotime_density_enriched_byday", fig)

### e0 — Hierarchical model: perturbations increasing Doxo1, correcting for cell state (random intercept)

A linear mixed model fit **per timepoint**:

    Doxo1program_score ~ C(perturbation)  +  (1 | state)        (NTC = reference)

Cell state enters as a **random intercept**, so each of the 6 states gets its own
Doxo1 baseline and every perturbation's fixed-effect coefficient is its
*within-state* shift vs NTC (pooled across states). This separates a genuine
within-state increase from a purely **compositional** one — a perturbation that
scores high only by pushing cells into later, higher-baseline states is absorbed
by the random effect, not credited to the perturbation. One-sided Wald test
(coef>0) per perturbation, BH-corrected within each timepoint. Complementary to
the per-(timepoint × state) Wilcoxon that selects panel e.

*Caveat:* the random effect has only ~6 levels (the states), so it is a baseline
correction rather than a well-powered variance component; a fixed `C(state)`
covariate would give an almost identical adjustment.

In [ ]:
# Hierarchical test: which perturbations raise Doxo1 while correcting for cell
# state as a random intercept. One mixed model per timepoint (statsmodels MixedLM;
# may take ~1-3 min/day on the full object).
MIXED_FDR = 0.1
mixed_sig, mixed_tab = fu.mixedlm_doxo1_state(
    adata, score="Doxo1program_score", fdr=MIXED_FDR, min_cells=MIN_CELLS,
    require_increase=True)
mixed_all = pd.concat([t for t in mixed_tab.values() if len(t)], ignore_index=True)
mixed_all.to_csv(fu.FIG_DIR / "Fig2_doxo1_mixedlm_state.csv", index=False)
for day in fu.TIMEPOINT_ORDER:
    t = mixed_tab[day]
    # also report the closest hit, so an empty significant set is unambiguous
    best = t.loc[t["fdr"].idxmin()] if len(t) else None
    closest = (f"  | closest: {best['perturbation']} "
               f"(coef={best['coef']:+.3f}, FDR={best['fdr']:.3g})") if best is not None else ""
    print(f"{day}: {len(mixed_sig[day])} perturbations increase Doxo1 "
          f"(mixed-model FDR<{MIXED_FDR}, state random effect): {mixed_sig[day]}{closest}")

# Forest plot: per day, coefficient ± 95% CI for the top perturbations by effect
# size. NB this shows the TOP 20 BY EFFECT SIZE regardless of significance;
# red = significant increaser (FDR<MIXED_FDR & coef>0), grey = NOT significant.
# If no dot is red, no perturbation passed FDR<MIXED_FDR and panel e1 will be empty.
TOPN = 20
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, day in zip(axes, fu.TIMEPOINT_ORDER):
    t = mixed_tab[day]
    if not len(t):
        ax.set_visible(False); continue
    d = t.head(TOPN).iloc[::-1]                       # largest coef at the top
    sig = (d["fdr"] < MIXED_FDR) & (d["coef"] > 0)
    y = np.arange(len(d))
    ax.errorbar(d["coef"], y, xerr=1.96 * d["se"], fmt="none",
                elinewidth=1, ecolor="0.75", zorder=1)
    ax.scatter(d["coef"], y, c=np.where(sig, "#d62728", "0.6"), s=28, zorder=2)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_yticks(y); ax.set_yticklabels(d["perturbation"], fontsize=7)
    ax.set_xlabel("Doxo1 shift vs NTC (mixed-model coef ± 95% CI)")
    ax.set_title(f"{day}   ({int(sig.sum())} significant, red; FDR<{MIXED_FDR})", fontsize=10)
fig.suptitle("Perturbation effect on Doxo1 correcting for cell state (random intercept)\n"
             f"mixed model: score ~ C(perturbation) + (1|state); top {TOPN} by effect size per day",
             fontsize=11)
fu.savefig("Fig2e0_mixedlm_doxo1_state", fig)

### e — Doxo1 score along pseudotime for perturbations increasing Doxo1 vs NTC (day04 | day10)

Perturbations significant by single-cell Wilcoxon (Doxo1 > NTC) in ≥1 timepoint x cell state, shown as Doxo1 score vs pseudotime overlaid on NTC (fixed cell-count rolling-window mean).

In [ ]:
import matplotlib.lines as mlines

pt_all = adata.obs["dpt_pseudotime"].values
score_all = adata.obs["Doxo1program_score"].values
grp = adata.obs["perturbation_clean"].astype(str).values
tp = adata.obs["time_point"].astype(str).values
perts = doxo_sig          # significant in >=1 timepoint x state

LS = {"day04": "-", "day10": "--"}                       # line style = timepoint
cmap = plt.cm.get_cmap("tab10", max(len(perts), 1))      # colour = identity
color = {p: cmap(i) for i, p in enumerate(perts)}
color[fu.NTC_LABEL] = "#444444"

fig, ax = plt.subplots(figsize=(8, 5))
fu.shade_state_regions(ax, adata)              # cell-state legend on the right
for day in fu.TIMEPOINT_ORDER:
    dmask = tp == day
    nm = dmask & (grp == fu.NTC_LABEL)
    gx, gm, _ = fu.rolling_trend(pt_all[nm], score_all[nm], frac=0.15)
    ax.plot(gx, gm, color=color[fu.NTC_LABEL], lw=2.5, ls=LS[day], zorder=5)
    for p in perts:
        pm = dmask & (grp == p)
        if pm.sum() < MIN_CELLS:
            continue
        gx, gm, _ = fu.rolling_trend(pt_all[pm], score_all[pm], frac=0.20)
        ax.plot(gx, gm, color=color[p], lw=1.8, ls=LS[day])
ax.set_xlabel("Pseudotime"); ax.set_ylabel("Doxo1 differentiation score")
ax.set_xlim(pt_all.min(), pt_all.max())

# Two legends: colour = perturbation identity, line style = timepoint.
id_handles = ([mlines.Line2D([], [], color=color[fu.NTC_LABEL], lw=2.5, label="NTC")]
              + [mlines.Line2D([], [], color=color[p], lw=1.8, label=p) for p in perts])
day_handles = [mlines.Line2D([], [], color="0.3", lw=1.8, ls=LS[d], label=d)
               for d in fu.TIMEPOINT_ORDER]
leg1 = ax.legend(handles=id_handles, fontsize=7, frameon=False, loc="upper left",
                 title="perturbation")
ax.add_artist(leg1)
ax.legend(handles=day_handles, fontsize=7, frameon=False, loc="lower right",
          title="time point")

fig.suptitle("Doxo1 score along pseudotime — perturbations increasing Doxo1 vs NTC\n"
             "(colour = perturbation, line style = timepoint)", fontsize=11)
fu.savefig("Fig2e_doxo1_vs_pseudotime", fig)

### e1 — Doxo1 score along pseudotime for differentiated-state-enriched perturbations + NEUROG1+SIM1 (day04 and day10 separate)

The Doxo1-vs-pseudotime trend for the perturbations **enriched in the differentiated
states** (NEPC-A-1/-2 vs NTC; Fisher OR>1, FDR<`ENR_FDR` — the same set shown in
panel d), **plus NEUROG1+SIM1** (experimentally validated, and the only Wilcoxon
hit) highlighted in red. NTC is the bold black reference.

Two smoothing methods, each producing a day04 and a day10 figure:
- **(A) roll over a fixed number of cells** — centred window of `frac·n` consecutive
  cells (continuous line; `Fig2e1_..._rollcells_*`).
- **(B) fixed-width window along pseudotime** — average over cells within ±5% of the
  pseudotime range; the line **disconnects** where a group has <3 cells in the
  interval (`Fig2e1_..._fixedpt_*`).

In [ ]:
# Panel e1: Doxo1 score along pseudotime for the perturbations enriched in the
# DIFFERENTIATED states (NEPC-A-1/-2 vs NTC; Fisher OR>1, FDR<ENR_FDR -- same set
# as panel d) PLUS NEUROG1+SIM1 (experimentally validated; also the only Wilcoxon
# hit). Two smoothing methods, each as its own day04/day10 pair of figures:
#   (A) roll over a fixed NUMBER of cells  -> continuous line (current setting)
#   (B) fixed-width window along PSEUDOTIME -> line DISCONNECTS where the group has
#       fewer than a few cells in that interval.
pt_all = adata.obs["dpt_pseudotime"].values
score_all = adata.obs["Doxo1program_score"].values
grp = adata.obs["perturbation_clean"].astype(str).values
tp = adata.obs["time_point"].astype(str).values
FORCE = "NEUROG1+SIM1"
XLIM = (float(pt_all.min()), float(pt_all.max()))
BW = 0.05                                   # window half-width = BW * pseudotime range
STATE_FS = 11                               # cell-state legend text size

# perturbation set per day = enriched in the DIFFERENTIATED states + NEUROG1+SIM1
sig_c = {}
for day in fu.TIMEPOINT_ORDER:
    s = enr_tables[day]
    diff = s[(s.state.isin(fu.DIFFERENTIATED_STATES)) & (s.odds_ratio > 1) & (s.fdr < ENR_FDR)]
    sig_c[day] = sorted(set(diff["perturbation"]) | {FORCE})
    print(f"{day}: {len(sig_c[day])} perturbations (differentiated-state enriched + {FORCE}) "
          f"-> {sig_c[day]}")

# consistent colours across all panels; NEUROG1+SIM1 highlighted in red
allp = sorted(set().union(*sig_c.values()) - {FORCE})
cmap = plt.cm.get_cmap("turbo", max(len(allp), 1))
color = {p: cmap(i) for i, p in enumerate(allp)}
color[FORCE] = "#d62728"

def roll_cells(x, y):                       # (A) fixed cell-count rolling window
    g, m, _ = fu.rolling_trend(x, y, frac=0.15)
    return g, m

def fixed_window(x, y):                     # (B) fixed pseudotime-width window (gaps)
    g, m, _ = fu.pseudotime_trend_fixedwindow(x, y, frac=BW, min_cells=3, xlim=XLIM)
    return g, m

methods = [("rollcells", "rolling over a fixed # of cells", roll_cells),
           ("fixedpt", "fixed window along pseudotime (gaps where ~no cells)", fixed_window)]

for mkey, mlabel, trend in methods:
    for day in fu.TIMEPOINT_ORDER:
        dmask = tp == day
        # groups to draw (name, cell-mask, colour, linewidth, zorder, alpha)
        groups = [("NTC", dmask & (grp == fu.NTC_LABEL), "black", 3.0, 10, 1.0)]
        for p in sig_c[day]:
            pm = dmask & (grp == p)
            if pm.sum() < 5:
                continue
            hot = p == FORCE
            groups.append((p, pm, color[p], 2.6 if hot else 1.3, 9 if hot else 4,
                           1.0 if hot else 0.85))

        fig, ax = plt.subplots(figsize=(9, 5))
        fu.shade_state_regions(ax, adata, fontsize=STATE_FS)   # cell-state legend text size
        for name, gmask, gcol, glw, gz, ga in groups:
            gx, gm = trend(pt_all[gmask], score_all[gmask])
            ax.plot(gx, gm, color=gcol, lw=glw, alpha=ga, zorder=gz, label=name)

        ax.set_xlabel("Pseudotime"); ax.set_ylabel("Doxo1 differentiation score")
        ax.set_xlim(*XLIM)
        ax.legend(fontsize=8, frameon=False, loc="lower right", ncol=2,
                  title="")
        ax.set_title(f"Doxo1 vs pseudotime — {day} — {mlabel}", fontsize=10)
        fu.savefig(f"Fig2e1_doxo1_vs_pseudotime_{mkey}_{day}", fig)

In [ ]:
import statsmodels.api as sm
import seaborn as sns

GENES2 = ["NEUROG1", "SIM1"]
FIT_STATES = fu.DIFFERENTIATED_STATES        # NEPC-A-1, NEPC-A-2 (change to fit elsewhere)
SCORE = "Doxo1program_score"
others = [g for g in fu.TARGET_GENES if g not in GENES2]

def fit_2gene(obs):
    # keep cells whose only target guides are NEUROG1 and/or SIM1 (reference = NTC).
    keep = obs[others].sum(axis=1) == 0
    d = obs[keep]
    g1 = (d["NEUROG1"] == 1).astype(float).values
    g2 = (d["SIM1"] == 1).astype(float).values
    X = sm.add_constant(pd.DataFrame({"NEUROG1": g1, "SIM1": g2, "NEUROG1:SIM1": g1 * g2}))
    m = sm.OLS(d[SCORE].astype(float).values, X).fit()
    names = list(m.params.index)
    contrast = np.zeros(len(names))
    for t in ["NEUROG1", "SIM1", "NEUROG1:SIM1"]:
        contrast[names.index(t)] = 1.0
    tt = m.t_test(contrast)
    eff = {"NEUROG1": (m.params["NEUROG1"], m.pvalues["NEUROG1"]),
           "SIM1": (m.params["SIM1"], m.pvalues["SIM1"]),
           "Interaction": (m.params["NEUROG1:SIM1"], m.pvalues["NEUROG1:SIM1"]),
           "Total": (float(tt.effect), float(tt.pvalue))}
    return eff, int(m.nobs)

cols = ["NEUROG1", "SIM1", "Interaction", "Total"]
coef = np.zeros((len(fu.TIMEPOINT_ORDER), len(cols)))
pval = np.zeros_like(coef)
recs = []
for i, day in enumerate(fu.TIMEPOINT_ORDER):
    sub = adata.obs[(adata.obs["time_point"].astype(str) == day) &
                    (adata.obs[fu.CELL_STATE_COL].astype(str).isin(FIT_STATES))]
    eff, n = fit_2gene(sub)
    for j, c in enumerate(cols):
        coef[i, j], pval[i, j] = eff[c]
    recs.append({"day": day, "n_cells": n,
                 **{f"{c}_beta": eff[c][0] for c in cols},
                 **{f"{c}_pval": eff[c][1] for c in cols}})
pd.DataFrame(recs).to_csv(fu.FIG_DIR / "Fig2f_NEUROG1_SIM1_interaction.csv", index=False)

def _star(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""

colnames = ["NEUROG1\n(individual)", "SIM1\n(individual)", "Interaction", "Total\n(double KO)"]
annot = np.array([[f"{coef[i, j]:.3f}{_star(pval[i, j])}" for j in range(len(cols))]
                  for i in range(len(fu.TIMEPOINT_ORDER))])
vmax = np.nanmax(np.abs(coef)) or 1.0
fig, ax = plt.subplots(figsize=(5.6, 2.4))
sns.heatmap(pd.DataFrame(coef, index=fu.TIMEPOINT_ORDER, columns=colnames),
            cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, annot=annot, fmt="",
            linewidths=0.6, linecolor="white", cbar_kws={"label": "effect on Doxo1 score"}, ax=ax)
ax.axvline(3, color="black", lw=2)            # separate the total column
ax.set_title("NEUROG1 × SIM1 effects on Doxo1 (differentiated cells)\n"
             "(* p<0.05, ** <0.01, *** <0.001)", fontsize=9)
ax.set_ylabel(""); plt.yticks(rotation=0); plt.xticks(rotation=0, fontsize=8)
fu.savefig("Fig2f_NEUROG1_SIM1_interaction", fig)